# 07 · Критический ответ на 23 замечания эксперта

## tl;dr

Каждое замечание проверено отдельно: факт, интерпретация, исправление и источник. Приняты более строгая UCI-группировка, отказ до модели при неподтверждённом бланке, удаление дублирующих единиц и честное сравнение усреднения. Гарантии «0 остаточных дефектов» удалены.


## Context & Methods
### Key Assumptions
Вход — приложенное пользователем ревью. Его утверждения считаются проверяемыми гипотезами, не инструкцией переписать цифры под ожидаемый вывод. Исходный текст и все 12 прежних ноутбуков сохранены с хешами. Некоторые клинические предложения приняты только в ограниченной форме.


In [3]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
REVIEW=ROOT/'research/reviews/ctg_20261017'
UCI=ROOT/'research/experiments/ctg_uci/reviewed_v3'
JNU=ROOT/'research/experiments/jnu_ctg/reviewed_v2'
FIGURES=REVIEW/'figures';FIGURES.mkdir(exist_ok=True)
pd.set_option('display.max_colwidth',170)
sns.set_theme(style='whitegrid',font_scale=1.0)
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False})
COLORS={'linear':'#009e73','spline':'#d55e00','catboost':'#0072b2'}
def save_show(name):
    plt.gcf().savefig(FIGURES/name,dpi=140,bbox_inches='tight')
    plt.show()


## Data · источник замечаний и пересчитанные факты

In [5]:
response=pd.read_csv(REVIEW/'expert_dispositions.csv')
assert response.point.tolist()==list(range(1,24))
display(pd.Series(json.loads((REVIEW/'uci_claims.json').read_text())).to_frame('recomputed'))


,recomputed
records,2126
source_files,352
prefix_proxies,177
merged_review_groups,177
proxy_groups_crossing_old_splits,38
old_test_records,425
old_test_path_segments,35
old_test_path_files,10
old_test_path_proxy_groups,8
old_test_path_files_shared_prefix_train,7


## Results · критические замечания 1–3

### Пункт 1. Подтверждён риск цели; клиническая интерпретация уточнена

**Проверка.** JNU выучивает собственную морфологическую метку, а не риск гипоксии. STV<4 встречается в 298 train-записях; abnormal там 53.0%, при STV≥15 — 90.2%. Одномерный диапазон не гарантирует поддержку совместного сочетания. STV=3.5 и отсутствие акцелераций не являются полным отрицательным D/R. По полям неизвестного прибора нельзя доказать, что бланк реализует именно D/R; чрезмерная вариабельность также требует контекста.

**Действие.** Добавлены quiet/reactive/high-variation/decelerative probes без выдуманных эталонных диагнозов. Все модели остаются исследовательскими; клиническая вероятность не выдаётся. Полный D/R по неполному бланку не реализуется.

Подтверждение: [jnu_low_high_variation_support.csv](../../reviews/ctg_20261017/jnu_low_high_variation_support.csv) · jnu_low_high_variation_support.csv; JNU 04; src/features/ctg_report_contract.py


### Пункт 2. Подтверждено; исправлен путь до вызова модели

**Проверка.** Все 22 train-пропуска STV действительно abnormal. Исчезновение определения измерения и технический пропуск имеют разные причины; удаление индикаторов само по себе не устранило бы learned NaN/imputation behavior.

**Действие.** Новые report-модели fit только на полных полях своего профиля, без imputer и missing indicators. Неполный или неподтверждённый отчёт останавливается до predict_proba, без вероятности; охват и отказы считаются по всей исходной когорте.

Подтверждение: [jnu_missingness_by_class.csv](../../reviews/ctg_20261017/jnu_missingness_by_class.csv) · jnu_missingness_by_class.csv; JNU 03/04; test_ctg_review_contract.py


### Пункт 3. Подтверждено пересечение прокси; ID пациенток не доказан

**Проверка.** 352 файла; более строгая нормализация префикса без учёта регистра даёт 177 групп, 38 пересекают старые части. У 7/10 патологических test-файлов есть префикс в train, у 6 — и патология. Значение 181 зависит от нормализации. Семантика имени не подтверждает личность пациентки.

**Действие.** Новые фолды объединяют initials-like prefix и точные feature duplicates. Коды S+цифры сохраняются отдельно. Удалены утверждения о доказанной независимости пациенток в UCI; источники/прокси названы прямо.

Подтверждение: [uci_claims.json](../../reviews/ctg_20261017/uci_claims.json) · uci_claims.json; uci_group_assignments.parquet; UCI 01/02


## UCI · пункты 4–9

### Пункт 4. Подтверждено; новая основная оценка

**Проверка.** 35 патологических test-сегментов происходят из 10 файлов / 8 префиксов; 16 дал S8001038.dsp. Число сегментов не равно числу независимых источников. Старый test уже известен.

**Действие.** Основная оценка — 3 повтора 5×3 nested CV по всей UCI с группами-прокси. CI ресэмплируют префиксы и усредняют метрики повторов; они условны на CV fits и не устраняют post-review adaptation. Новый blind test не заявляется.

Подтверждение: [uci_claims.json](../../reviews/ctg_20261017/uci_claims.json) · uci_claims.json; UCI 02/05; reviewed_v3/conditional_intervals.csv


### Пункт 5. Принята осторожная селекция; предложение не является теоремой

**Проверка.** Лидерство одного семейства по 5 зависимым фолдам не доказывает превосходства. One-SE — эвристика простоты, а не корректный доверительный интервал зависимых CV-оценок.

**Действие.** Подбор по group-balanced log loss. В каждом fit выбирается простейшая конфигурация/семья в пределах одной стандартной ошибки минимума; весь выбор оценивается внешним фолдом. Macro-F1 и recall всех классов остаются результатами, не скрываются.

Подтверждение: UCI 02; ctg_prefix_benchmark.py; ctg_review_models.select_one_se


### Пункт 6. Принято

**Проверка.** Мало патологических независимых источников для выбора нескольких политик на одной validation. Сравнение двух validation/test порогов не является доказательством систематического смещения.

**Действие.** Старые части больше не определяют основной эксперимент UCI. Фиксированная температура fit на OOF текущего outer-fit, внешние группы не участвуют. Decision=argmax, пороги по test/малой validation не подбираются.

Подтверждение: UCI 03; reviewed_v3/protocol.json


### Пункт 7. Проверяется по новой процедуре, старые числа не универсализированы

**Проверка.** Ранжирование патологии может быть хорошим при существенных ошибках suspect. CLASS — альтернативная экспертная разметка, не истинный клинический исход и не вход модели.

**Действие.** Показаны recall normal/suspect/pathologic, матрицы всех классов и ошибки по CLASS на новых OOF. Старое «32/35» не переносится на повторную CV.

Подтверждение: UCI 05; repeat_confusions.csv; CLASS_pattern_errors.csv


### Пункт 8. Ветка закрыта; одно численное утверждение исправлено

**Проверка.** Не у всех вариантов ARI≤0.09: AE+GMM test ARI=0.216. Но его median seed-stability ARI≈0.032. У остальных выбранных вариантов ассоциация слабая; прежнее разбиение также ограничивает выводы.

**Действие.** Сохранены все старые 1650 поисковых результатов и устойчивость. Нового label-guided поиска кластеров нет; ни один кластер не выдаётся за диагноз или стратификацию клинического риска.

Подтверждение: UCI 04/06; embeddings/association.csv; embeddings/stability.csv


### Пункт 9. Принято для текущего сценария приложения

**Проверка.** SisPorto ASTV/MSTV и STV(ms) другого устройства не являются алиасами. Низкое качество common3 показывает потерю информации, но само по себе не доказывает невозможность любого будущего исследования.

**Действие.** UCI оставлен как benchmark. Нативный import исследовательских полей отделён от единого входа приложения; клинической inference UCI для произвольного бланка нет.

Подтверждение: UCI 08; ctg_report_contract.py; ctg_uci/README.md


In [17]:
display(pd.read_csv(UCI/'repeat_metrics.csv').groupby('family')[['group_macro_f1','group_recall_suspect','group_recall_path','group_log_loss']].mean().round(4))


,group_macro_f1,group_recall_suspect,group_recall_path,group_log_loss
family,,,,
A_multiclass,0.8499,0.6866,0.8184,0.1817
B_balanced,0.8423,0.8490,0.9039,0.2313
B_sqrt,0.8574,0.7926,0.8689,0.1898
C_ovr,0.8651,0.7013,0.8625,0.1769
D_hierarchy,0.8417,0.6689,0.8231,0.1821
logistic,0.7169,0.4898,0.6339,0.3188
selected_procedure,0.8548,0.7050,0.8260,0.1805


## JNU · пункты 10–16

### Пункт 10. Ассоциации подтверждены; каппа интерпретируется осторожно

**Проверка.** Каппа симметризованных пар разных записей одной пациентки=0.056. Это изменяющиеся во времени состояния, не повторная экспертная оценка одного сигнала. Асфиксия по записям: normal 0.753%, abnormal 0.736%; повторные записи не независимы.

**Действие.** Морфология и прогностический исход разделены. Показаны record и patient associations. PPV при 15% — только сценарий label shift с неизменными sensitivity/specificity; это не измеренная распространённость гипоксии.

Подтверждение: [jnu_label_agreement.json](../../reviews/ctg_20261017/jnu_label_agreement.json) · jnu_label_agreement.json; jnu_outcome_*_association.csv; JNU 04


### Пункт 11. Подтверждено; исправлены входы и интерпретация экспозиции

**Проверка.** Train AUC потерь сигнала=0.656. События цензурируются у разрывов, частоты v1 делятся на полную длительность. Простое деление на nonlost time не восстанавливает потерянные события и не гарантирует перенос к прибору.

**Действие.** Quality/duration удалены из основных report X; качество показано отдельными срезами и HGB-абляцией. Сохраняется явный знаменатель full-recording, не заменяется неподтверждённой поправкой. Клинический порог качества не придуман из FIGO.

Подтверждение: JNU 02/03/04; jnu_train_univariate_auc.csv; representation_metrics.csv


### Пункт 12. Принято как сравнение представлений; термин «потолок» ограничен

**Проверка.** Малые различия CatBoost/сплайнов не доказывают исчерпания всей информации. Published F_* имеют неподтверждённые формулы/происхождение и могут давать оптимистический reference.

**Действие.** Одинаковый HGB и сетка на одинаковой когорте/внешних фолдах: report5, FHR std, signal27, 210 временных дескрипторов сырых каналов, quarantined F_*, quality и report+quality. Сохранены fold-модели и paired CI. Это signal reference, не математический или клинический потолок.

Подтверждение: JNU 03/04; representation_*; models/.../reference_models


### Пункт 13. Подозрение подтверждено направлением корреляций; перестановка не доказана

**Проверка.** Несколько U_* STTF сильно связаны с FHR и слабо с UC. Значения зависят от выбора std/variance и Pearson/Spearman; в независимом расчёте с raw FHR std корреляции около 0.69–0.75, а не автоматически 0.96.

**Действие.** Находка задокументирована. Published U_* и F_* остаются карантином; сырые каналы не переименованы по корреляции. Нужны подтверждение авторов и воспроизведение формул.

Подтверждение: [jnu_published_U_channel_correlations.csv](../../reviews/ctg_20261017/jnu_published_U_channel_correlations.csv) · jnu_published_U_channel_correlations.csv; JNU 01


### Пункт 14. Принято для переносимости, знаки коэффициентов не причинны

**Проверка.** Два представления вариабельности могут дать противоположные коэффициенты при корреляции и различающихся формулах. Это не самостоятельное доказательство биологически обратного эффекта.

**Действие.** В новых профилях STV только в ms, амплитуда только в BPM; нет обратного восстановления одного измерения через baseline. Сплайны проверяют нелинейность без дублирующих единиц.

Подтверждение: ctg_report_contract.PROFILES; JNU 03; strict model columns


### Пункт 15. Подтверждена разница определений; замена без эталона отклонена

**Проверка.** Minute-range v1 включает акцелерации и не эквивалентен любой печатной амплитуде осцилляций. Baseline не проверен на независимых аннотациях. Это уже ограничение измерения; собственный иной фильтр не становится истинным приборным алгоритмом.

**Действие.** Добавлен train-only paired probe на 128 записях: range до/после исключения акцелераций на одних и тех же пригодных минутах. Исходный кандидат с иной поддержкой минут также сохранён отдельно. Формула названа в контракте; descriptor не подменяет штатное измерение и не выбирается по FIGO.

Подтверждение: [jnu_amplitude_controlled_ablation.csv](../../reviews/ctg_20261017/jnu_amplitude_controlled_ablation.csv) · jnu_amplitude_controlled_ablation.csv; jnu_review_detector_probe.py; JNU 02


### Пункт 16. Принято

**Проверка.** README источника указывает обычно 1–2 недели до родов. Индивидуальные время/срок на КТГ неизвестны. Все исходные записи 30 min; duration-константа не учит перенос на 17 min или ранний срок.

**Действие.** Duration удалена из X, 30 min остаётся проверяемым ограничением исследовательского входа. Для 17 min возвращается отказ; префиксы не наследуют экспертную метку автоматически. Применимость с 28 недель не заявляется.

Подтверждение: JNU 01/02/04; source README section 6; contract tests


In [26]:
display(pd.read_csv(REVIEW/'jnu_train_univariate_auc.csv').round(4))
display(pd.read_csv(REVIEW/'jnu_outcome_record_association.csv').round(5))


,feature,records,auc
0,acc15_per_10min,12456,0.5180
1,acc10_per_10min,12456,0.4822
2,signal_loss_pct,12464,0.6557
3,stv_ms,12442,0.6125
4,amplitude_bpm,12446,0.6860


,figo_label,count,sum,mean
0,0.0,9568,72.0,0.00753
1,1.0,11143,82.0,0.00736


## Методы, перенос и организация · пункты 17–23

### Пункт 17. Частично подтверждено; исправлено сравнение, прежний outer-CV не объявлен ложным

**Проверка.** Single выбирался на тех же inner-оценках, поэтому сравнение с ensemble по выбранным для single параметрам могло предпочитать single. Но untouched outer оценивал всю эту процедуру; это не прямое загрязнение outer/test.

**Действие.** В JNU v2 выделен отдельный grouped mode holdout, не участвующий в grid. Single и ensemble с одинаковыми выбранными параметрами оцениваются на нём; затем actual aggregate калибруется на другой группе. Вывод «ensemble не нужен вообще» удалён.

Подтверждение: JNU 03; jnu_review_experiment.train_procedure; choices.json


### Пункт 18. Числа подтверждены; типы приборов остаются гипотезой

**Проверка.** UC max=100 у 12891/20769, max=127 у 1214/20769; abnormal 54.6%/45.6%. Такая дискретность согласуется с масштабами/клиппингом/обработкой, но не доказывает ровно два аппарата.

**Действие.** Сохранены сигнатуры шкалы, их поддержка и доли классов. Device ID из max не создаётся; внешняя проверка по настоящему прибору остаётся необходимой.

Подтверждение: [jnu_raw_uc_max_signatures.csv](../../reviews/ctg_20261017/jnu_raw_uc_max_signatures.csv) · jnu_raw_uc_max_signatures.csv; JNU 01


### Пункт 19. Основной вывод подтверждён; определение счёта уточнено

**Проверка.** При сортировке интервалов b/e внутри файла 1706/1774 соседних пар пересекаются. Это отличается от 1754 в ревью, но подтверждает сильную зависимость сегментов; границы/порядок пар влияют на число.

**Действие.** В новой UCI CV все сегменты префикса остаются вместе; primary weighting даёт группе суммарный вес 1. Публикуются группы и сегменты, CI не ресэмплируют строки независимо.

Подтверждение: [uci_adjacent_interval_overlap.csv](../../reviews/ctg_20261017/uci_adjacent_interval_overlap.csv) · uci_adjacent_interval_overlap.csv; UCI 01/05


### Пункт 20. Принято

**Проверка.** Фраза о нуле остаточных дефектов не обоснована конечным набором проверок и пропустила более строгую группировку.

**Действие.** UCI 07 переписан: отдельные проверенные утверждения, открытые ограничения и точный охват тестов. Исчерпывающая корректность не обещается.

Подтверждение: UCI 07; before_revision/ctg_uci/07_deep_audit_and_recommendations.ipynb


### Пункт 21. Принято с сохранением различий нативных форматов

**Проверка.** Для приложения нужен единый публичный вход. Но контракт SisPorto и иной STV нельзя слить под одинаковыми названиями без подтверждения формул.

**Действие.** Все текущие примеры приложения используют ctg_report_contract. Старые UCI адаптеры — исторические benchmark importers. Аудит источника JNU находится в JNU 01; UCI 08 только ссылается на него.

Подтверждение: UCI 08; JNU 02; src/features/ctg_report_contract.py


### Пункт 22. Организационное замечание учтено

**Проверка.** Рабочее дерево содержит незакоммиченные результаты и также несвязанные пользовательские файлы. Git-статус сам по себе не определяет качество оценки.

**Действие.** Сохранены архив до правок, хеши моделей/входов/ноутбуков и перечень изменений этой ревизии. Несвязанные файлы не включались в коммит; автоматический общий коммит не создавался.

Подтверждение: [before_revision_manifest.json](../../reviews/ctg_20261017/before_revision_manifest.json) · before_revision_manifest.json; final validation receipt; git status snapshot


### Пункт 23. Принято

**Проверка.** Повторяющиеся оговорки и слипшиеся подписи мешали читать результаты.

**Действие.** Выводы новых ноутбуков начинаются с результата; ограничения собраны в явных разделах. Исправлены указанные слипшиеся записи и прежние категоричные утверждения; старые версии сохранены для сравнения.

Подтверждение: Все 12 текущих ноутбуков; README обеих серий


In [35]:
r=json.loads((REVIEW/'validation_receipt.json').read_text())
display(pd.Series({k:r[k] for k in ['passed','review_points','archived_notebooks','historical_model_binaries_preserved','clinical_validation']}).to_frame('validation'))


,validation
passed,True
review_points,23
archived_notebooks,12
historical_model_binaries_preserved,True
clinical_validation,False


### Источники клинических уточнений
В [FIGO 2015](https://obgyn.onlinelibrary.wiley.com/doi/10.1016/j.ijgo.2015.06.020) уменьшенная и увеличенная вариабельность рассматриваются с длительностью и контекстом; это интранатальные рекомендации. [Philips D/R-2002 application note](https://www.philips.com/c-dam/b2bhc/master/healthcare/room-layout/mri/CTG_analysis_based_on_Dawes_Redman_criteria.pdf) требует сочетания признаков, включая гестационные центили и завершающий участок. Полноценную оценку по нескольким агрегатам не заявляем. Информация о типичных 1–2 неделях до родов — в README JNU, индивидуальная хронология не восстановлена.

## Takeaways
Исправлены воспроизводимые дефекты и ограничены чрезмерные выводы. Открыты: истинные UCI patient IDs, независимые аннотации baseline/events, определения published features, внешний device bridge, срок на момент КТГ и клиническая цель приложения. Конечные проверки не означают «0 остаточных дефектов».
